# 04 · Residuos y evaluación de alertas

**Clases 4, 9 y 12:** boxplot, matriz de confusión y diagnóstico de residuos.

El regressor predice un número continuo. La regla de boxplot señala días con residuo relativo menor que Q1 − 1,5 × IQR. El umbral se calcula por planta usando predicciones fuera de muestra de entrenamiento (cinco folds agrupados por mes).

Las pruebas de caída de 10%, 25%, 50% y 100%, durante 1, 3 y 7 días, se aplican solamente después del entrenamiento. No se entrena con datos sintéticos. Es un benchmark controlado: las etiquetas conocidas son las inyecciones, no fallas operativas confirmadas.

Cada episodio tiene un identificador único. Los días sin decisión se cuentan aparte y no entran como verdaderos negativos.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
import json
from src.academico import metricas_confusion
E=json.loads((DATA_PROCESSED/'defensa_evidencia.json').read_text())
sims=pd.DataFrame(E['simulaciones'])
assert sims.id.is_unique
filas=[]
for caida,g in sims.groupby('caida'):
    m=metricas_confusion(**g[['tn','fp','fn','tp']].sum().astype(int).to_dict())
    filas.append({'caida':caida,**m,'episodios':len(g),'sin_decision':g.sin_decision.sum(),
                  'episodios_detectados':g.demora.notna().sum(),'demora_mediana':g.demora.median()})
tabla=pd.DataFrame(filas)
print(tabla.round(4).to_string(index=False))


In [ ]:
g=tabla.loc[tabla.caida.eq(.5)].iloc[0]
fig=go.Figure(go.Heatmap(z=[[g.tn,g.fp],[g.fn,g.tp]],x=['Predijo normal','Predijo alerta'],
                         y=['Real normal','Real caída'],texttemplate='%{z}',colorscale='Blues',showscale=False))
fig.update_layout(title='Clase 9 · Matriz de confusión: caída controlada del 50%')
fig.show()
px.line(tabla,x='caida',y=['precision','recall','f1'],markers=True,
        title='Clase 9 · Precision, recall y F1 por magnitud de caída').show()
print('Alcance sin inyección:',E['regresion']['baseline_sin_inyeccion'])


## Interpretación y límites

Precision responde cuántas alertas coinciden con las caídas inyectadas; recall, cuántos días con caída detecta. Accuracy puede parecer alta por la abundancia de días sin caída. La proporción de positivos la fija el experimento: no se interpreta precision como una probabilidad garantizada en campo.

No se decide si el sensor está caído, hay registros inválidos o la energía esperada es demasiado pequeña (regla de calidad del proyecto). station04 tiene capacidad no confiable. Las alertas de la historia son indicios de revisión, no diagnósticos de causas físicas.
